In [1]:
import pandas as pd 
dataset=pd.read_csv('100_Unique_QA_Dataset.csv')

In [2]:
dataset.head(3)

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee


In [3]:
len(dataset)

90

In [4]:
def tokenize(text):
    text=text.lower()
    text=text.replace('?'," ")
    text=text.replace("'"," ")
    return text.split()

In [5]:
vocab={'<UNK>':0}

In [6]:
len(vocab)

1

In [7]:
def build_vocab(row):
    tokenized_question=tokenize(row['question'])
    tokenized_output=tokenize(row['answer'])
    merged_token=tokenized_question+tokenized_output
    for token in merged_token:
        if token not in vocab:
            vocab[token]=len(vocab)
    

In [8]:
dataset.apply(build_vocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [9]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [10]:
len(vocab)

324

In [11]:
#Text to indices 
def text_to_indeces(text,vocab):
    indexed_text=[]
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [12]:
text_to_indeces('What is nikhil',vocab)

[1, 2, 0]

In [13]:
import torch
from torch.utils.data import Dataset,DataLoader

In [14]:
class QADataset(Dataset):
    def __init__(self,df,vocab):
        self.df=df
        self.vocab=vocab
    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self,index):
        x=text_to_indeces(self.df.iloc[index]['question'],self.vocab)
        y=text_to_indeces(self.df.iloc[index]['answer'],self.vocab)
        return torch.tensor(x),torch.tensor(y)

In [15]:
datasets=QADataset(dataset,vocab)

In [16]:
datasets[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [17]:
datasets.__len__()

90

In [18]:
dataloader=DataLoader(
    datasets,
    batch_size=1,
    shuffle=True
)

In [19]:
for ques,answer in dataloader:
    print(ques,answer)

tensor([[10, 55,  3, 56,  5, 57]]) tensor([[58]])
tensor([[ 42, 101,   2,   3,  17]]) tensor([[102]])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([[215]])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor([[131]])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([[36]])
tensor([[ 42, 137,   2, 138,  39, 175, 269]]) tensor([[99]])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([[95]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]])
tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[260]])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([[254]])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([[298]])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([[285]])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([[100]])
tensor([[ 42, 107,   2, 108,  19, 109]]) tensor([[110]])
tensor([[  1,   2,   3, 221,   5, 222, 223, 224]]) tensor([[225]])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([[205]])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([[238]])
ten

In [20]:
import torch.nn as nn

In [21]:
class SimpleRnn(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 50)
        self.Rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, x):
        x = self.embedding(x)

        hidden, final = self.Rnn(x)

        final = final.squeeze(0)

        output = self.fc(final)

        return output

In [22]:
embedding=nn.Embedding(len(vocab),50)

In [23]:
embedding

Embedding(324, 50)

In [24]:
a=embedding(datasets[0][0])

In [25]:
datasets[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [26]:
a

tensor([[-0.7505,  0.0225,  0.7775,  0.8341, -0.6705, -0.6747,  1.6049, -0.6536,
          0.4337,  1.5497,  1.3976, -0.1299,  1.1335, -1.4980, -0.3913, -0.4940,
         -0.5528, -0.3040, -1.1271, -1.2377,  0.8178,  1.2343,  0.3583,  0.2692,
          0.4138,  0.1110, -1.3484,  1.0036, -0.9149,  1.1230,  0.1858, -0.2760,
         -0.8527, -2.4908, -1.1274, -1.9989,  0.0193, -0.7731, -0.7915,  1.3997,
         -0.0802,  0.1844,  1.2217,  1.0789,  0.2006,  0.9317, -0.7900, -0.7885,
          0.5032,  1.4303],
        [-1.8111,  0.8798,  1.2828, -0.4334,  1.3450, -1.7875, -1.0619, -0.0362,
         -0.2236, -0.7950,  0.3265,  1.7641, -0.8463, -1.0182, -1.2312,  0.5906,
         -1.0690, -0.7349,  1.3574,  0.9400,  0.4548, -0.5311, -0.4952,  0.6460,
          1.8808, -0.0048, -0.5028,  1.4992,  0.7244,  0.0748, -1.8326, -0.3226,
         -0.0977, -1.3379, -0.7669, -0.7015, -1.1290,  1.9342, -0.1328,  0.0426,
         -0.3983, -0.7028, -0.8527, -0.2616, -0.7139, -1.5813,  0.3687,  1.2383,


In [27]:
b=nn.RNN(50,64)

In [28]:
#Hidden unit 
b(a)[0]

tensor([[ 0.4744,  0.6237, -0.7125,  0.5808, -0.6486,  0.2383, -0.0913, -0.3098,
          0.7967,  0.2953,  0.0404,  0.7426, -0.4452, -0.3176, -0.3197, -0.3550,
          0.0224,  0.3974, -0.4922,  0.3337,  0.1568,  0.7723,  0.2547, -0.1281,
         -0.2815,  0.7447, -0.0336,  0.3613, -0.0524,  0.5229, -0.8144,  0.4869,
          0.4055, -0.2907, -0.7430, -0.1629, -0.0226,  0.3543, -0.0800,  0.0627,
          0.2110,  0.3052, -0.2405,  0.0918, -0.2205, -0.1484,  0.4972, -0.1877,
         -0.1770,  0.4645, -0.3323, -0.2744,  0.3716,  0.4621, -0.4504,  0.1450,
         -0.4714,  0.4610,  0.1255,  0.0164,  0.5746,  0.8062,  0.3909,  0.1135],
        [ 0.4096,  0.1388, -0.0521,  0.4672, -0.6802, -0.6221,  0.5338, -0.5595,
          0.4272, -0.7673, -0.9104,  0.2255, -0.5958, -0.3813,  0.0193,  0.6454,
          0.2563, -0.3916,  0.2411, -0.0330,  0.6363,  0.3096, -0.5256, -0.0856,
         -0.6027, -0.5140,  0.2895,  0.4813,  0.1648, -0.4316, -0.1926,  0.8021,
          0.5289, -0.1100, 

In [29]:
#Final Output 
u=b(a)[1]

In [30]:
z=nn.Linear(64,len(vocab))

In [31]:
z(u)

tensor([[-0.0686,  0.0812, -0.2225, -0.0543,  0.1235, -0.3204,  0.1038, -0.1148,
         -0.0724,  0.1113,  0.0475, -0.2167, -0.0418,  0.0333,  0.4396,  0.2541,
          0.1708,  0.0151, -0.0627,  0.5445, -0.1974, -0.4595,  0.3057, -0.1096,
          0.1394,  0.1863,  0.1834, -0.1716,  0.1965,  0.2486, -0.1757, -0.7214,
          0.0633, -0.0402, -0.1507,  0.1698, -0.3770, -0.1852,  0.4446, -0.3093,
          0.4818,  0.1156, -0.4453,  0.3453,  0.0622,  0.1087,  0.3801, -0.0064,
         -0.1964,  0.4527, -0.1256, -0.0754,  0.3955,  0.4599, -0.2193, -0.0166,
          0.2954, -0.1447,  0.3165, -0.2129,  0.3778,  0.0987,  0.1115, -0.2448,
          0.0537, -0.0553,  0.3153, -0.2763,  0.2929, -0.3741, -0.1174,  0.4926,
          0.1033,  0.2284, -0.2461,  0.0048,  0.2711,  0.1743, -0.1783, -0.5361,
         -0.1944, -0.5074,  0.5635, -0.1442,  0.0892, -0.1972, -0.2096, -0.0444,
         -0.5190, -0.3656, -0.4422,  0.2228, -0.3653,  0.1615,  0.3681,  0.1160,
         -0.1723, -0.2397,  

In [32]:
z(u).shape

torch.Size([1, 324])

In [33]:
model=SimpleRnn(len(vocab))

In [34]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

In [35]:
ques, answer = next(iter(dataloader))

y_logits = model(ques)

print(ques.shape)
print(y_logits.shape)
print(answer.shape)

torch.Size([1, 7])
torch.Size([1, 324])
torch.Size([1, 1])


In [36]:
epochs = 20

for epoch in range(epochs):
    total_loss = 0

    model.train()

    for ques, answer in dataloader:

        y_logits = model(ques)

        train_loss = loss_fn(
            y_logits,
            answer.squeeze(1)
        )

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        total_loss += train_loss.item()

    print(f"Epoch: {epoch}, Loss: {total_loss}")

Epoch: 0, Loss: 537.4923994541168
Epoch: 1, Loss: 330.5655884742737
Epoch: 2, Loss: 162.55887615680695
Epoch: 3, Loss: 86.1342390589416
Epoch: 4, Loss: 39.81415587477386
Epoch: 5, Loss: 34.92813820205629
Epoch: 6, Loss: 27.00577942468226
Epoch: 7, Loss: 22.049778066575527
Epoch: 8, Loss: 16.559915273915976
Epoch: 9, Loss: 12.449429975822568
Epoch: 10, Loss: 8.181008069775999
Epoch: 11, Loss: 7.490333889611065
Epoch: 12, Loss: 4.42065531690605
Epoch: 13, Loss: 10.236999571789056
Epoch: 14, Loss: 4.703111701412126
Epoch: 15, Loss: 5.589746406767517
Epoch: 16, Loss: 2.6158011096995324
Epoch: 17, Loss: 3.1879369071684778
Epoch: 18, Loss: 14.449196055531502
Epoch: 19, Loss: 93.85768997343257
